In [1]:
from workspace.sources.local_datasets.preprocessing.tokenization import NLTKTokenizer
from workspace.sources.local_datasets.preprocessing.cleaning import NoiseReduction, Lemmatization, Stemming
from workspace.sources.experiments.metrics import Loss
from workspace.sources.experiments.experiment import Experiment
from workspace.sources.local_datasets.dataset import Dataset
from workspace.sources.utils import class_name_to_str
from workspace.sources.local_datasets.preprocessing.pipelines import PreprocessingPipeline
from workspace.sources.local_datasets.preprocessing.utils import truncate
from sklearn.model_selection import ParameterGrid
from IPython.display import display, Markdown

from workspace.sources.local_datasets.preprocessing.encoders.encoders import BertBaseUncasedEncoder as Encoder
from workspace.sources.models.transformers.bert_based_models import BERT as Model

import mlflow

mlflow.set_tracking_uri('../../mlruns')

In [2]:
experiment_name = f'prefinal-{class_name_to_str(Model.__name__)}-v2'
print('Experiment:', experiment_name)

Experiment: prefinal-bert-v2


In [3]:
def conduct_model_experiments(dataset_name,
                              dataset_path,
                              dataset_hparams_grid,
                              model_hparams_grid):
    total_runs = len(dataset_hparams_grid) * len(model_hparams_grid)
    for i, dataset_params in enumerate(dataset_hparams_grid):
        for j, model_hparams in enumerate(model_hparams_grid):
            run_number = i * len(model_hparams_grid) + j + 1
            display(Markdown(f'### Run {run_number} of {total_runs}'))
            dataset = Dataset(dataset_name, dataset_path, **dataset_params)

            model = Model(train_best_model_metric=Loss,
                          training_arguments=model_hparams)

            recovery_experiment = Experiment(
                name=experiment_name,
                dataset=dataset,
                model=model)
            recovery_experiment.run()


In [4]:

# max_tokens_params = [128, 512]
max_tokens_params = [128]

pipelines = []

for max_tokens in max_tokens_params:
    pipelines.extend([PreprocessingPipeline(name='minimal',
                                            iterable=[Encoder(truncation_max_length=max_tokens)]),
                      PreprocessingPipeline(name='noise_reduction',
                                            iterable=[NoiseReduction(),
                                                      Encoder(truncation_max_length=max_tokens)]),
                      PreprocessingPipeline(name='noise_reduction_with_lemmatization',
                                            iterable=[NoiseReduction(), NLTKTokenizer(), Lemmatization(),
                                                      Encoder(is_split_into_words=True,
                                                              truncation_max_length=max_tokens)]),
                      PreprocessingPipeline(name='noise_reduction_with_stemming',
                                            iterable=[NoiseReduction(), NLTKTokenizer(), Stemming(),
                                                      Encoder(is_split_into_words=True,
                                                              truncation_max_length=max_tokens)])
                      ])
# optional - for testing purposes, if you want to run fast test on very small datasets
truncate(pipelines, fraction=0.05)

dataset_hparams_grid = ParameterGrid({'preprocessings_pipeline': pipelines})

print('Dataset Hyperparameters Grid Size: ', len(dataset_hparams_grid))

# model_hparams_grid = ParameterGrid({'epochs': [10],
#                                     'batch_size': [16],
#                                     'learning_rate': [1e-05, 5e-05],
#                                     'lr_scheduler': ['linear'],
#                                     'optimizer': ['adamw_torch'],
#                                     'weight_decay': [0.0, 1e-3],
#                                     'early_stopping_patience': [3],
#                                     'early_stopping_threshold': [0.01]})

model_hparams_grid = ParameterGrid({'epochs': [10],
                                    'batch_size': [16],
                                    'learning_rate': [5e-05],
                                    'lr_scheduler': ['linear'],
                                    'optimizer': ['adamw_torch'],
                                    'weight_decay': [1e-3],
                                    'early_stopping_patience': [3],
                                    'early_stopping_threshold': [0.01]})

print('Model Hyperparameters Grid Size: ', len(model_hparams_grid))
print('Total Hyperparameter Combinations: ', len(dataset_hparams_grid) * len(model_hparams_grid))

Dataset Hyperparameters Grid Size:  4
Model Hyperparameters Grid Size:  1
Total Hyperparameter Combinations:  4


### ReCOVery Dataset

In [5]:
dataset_name = 'ReCOVery'
dataset_path = '../../sources/local_datasets/data/ReCOVery/recovery.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          dataset_hparams_grid,
                          model_hparams_grid)

### Run 1 of 4

2025-05-16 02:28:11,714 - Experiment - INFO - MLflow experiment initialized with ID: 102456789287063802
2025-05-16 02:28:11,715 - Experiment - INFO - Model signature: model(mn=bert,ti=bert-base-uncased,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01)
2025-05-16 02:28:11,716 - Experiment - INFO - Dataset signature: dataset(dn=recovery,tp=0.7,vp=0.15);pipeline([truncation(f=0.05,tt=top),bert_base_uncased_encoder(t=bert-base-uncased,t=true,p=max_length,tml=128,isiw=false)])
2025-05-16 02:28:11,717 - Experiment - INFO - Run signature: model(mn=bert,ti=bert-base-uncased,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01);dataset(dn=recovery,tp=0.7,vp=0.15);pipeline([truncation(f=0.05,tt=top),bert_base_uncased_encoder(t=bert-base-uncased,t=true,p=max_length,tml=128,isiw=false)])
2025-05-16 02:28:12,516 - Experiment - INFO - Run ID: 6ec495d23b4f4cafbd095713d8821845
2025-05-16 02:28:12,517 - Experiment - INFO - Run name: mode

Map:   0%|          | 0/71 [00:00<?, ? examples/s]

Map:   0%|          | 0/71 [00:00<?, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

...

2025-05-16 02:30:17,723 - Experiment - INFO - Test metrics: {'test_loss': 0.6419464945793152, 'test_accuracy': 0.6666666666666666, 'test_precision': 0.6666666666666666, 'test_recall': 1.0, 'test_f1_score': 0.8, 'test_roc_auc': 0.58, 'test_false_positives_rate': 1.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.163, 'test_samples_per_second': 92.04, 'test_steps_per_second': 6.136, 'test_epoch': 1.0}
2025-05-16 02:30:17,740 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:30:17,742 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:30:17,784 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:30:17,785 - Experiment - INFO - Evaluating model using f1_score metric...
2025-05-16 02:30:17,785 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-5
2025-05-16 02:30:18,189 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.7323901653289795, 'eval_accuracy': 0.4666666666666667, 

2025-05-16 02:30:18,336 - Experiment - INFO - Test metrics: {'test_loss': 0.6419464945793152, 'test_accuracy': 0.6666666666666666, 'test_precision': 0.6666666666666666, 'test_recall': 1.0, 'test_f1_score': 0.8, 'test_roc_auc': 0.58, 'test_false_positives_rate': 1.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.1422, 'test_samples_per_second': 105.499, 'test_steps_per_second': 7.033, 'test_epoch': 1.0}
2025-05-16 02:30:18,354 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:30:18,356 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:30:18,399 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:30:18,400 - Experiment - INFO - Evaluating model using false_positives_rate metric...
2025-05-16 02:30:18,400 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-5
2025-05-16 02:30:18,837 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.7323901653289795, 'eval_accuracy': 0.466

2025-05-16 02:30:18,979 - Experiment - INFO - Test metrics: {'test_loss': 0.6419464945793152, 'test_accuracy': 0.6666666666666666, 'test_precision': 0.6666666666666666, 'test_recall': 1.0, 'test_f1_score': 0.8, 'test_roc_auc': 0.58, 'test_false_positives_rate': 1.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.1343, 'test_samples_per_second': 111.649, 'test_steps_per_second': 7.443, 'test_epoch': 1.0}
2025-05-16 02:30:18,996 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:30:18,998 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:30:19,038 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:30:19,038 - Experiment - INFO - Evaluating model using roc_auc metric...
2025-05-16 02:30:19,039 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-10
2025-05-16 02:30:19,474 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.8063422441482544, 'eval_accuracy': 0.466666666666666

2025-05-16 02:30:20,033 - Experiment - INFO - Test metrics: {'test_loss': 0.6391021013259888, 'test_accuracy': 0.6666666666666666, 'test_precision': 0.6666666666666666, 'test_recall': 1.0, 'test_f1_score': 0.8, 'test_roc_auc': 0.56, 'test_false_positives_rate': 1.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.5489, 'test_samples_per_second': 27.329, 'test_steps_per_second': 1.822, 'test_epoch': 2.0}
2025-05-16 02:30:20,102 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:30:20,110 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:30:20,311 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:30:20,313 - Experiment - INFO - Evaluating model using loss metric...
2025-05-16 02:30:20,318 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-5
2025-05-16 02:30:21,731 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.7323901653289795, 'eval_accuracy': 0.4666666666666667, 'e

2025-05-16 02:30:22,042 - Experiment - INFO - Test metrics: {'test_loss': 0.6419464945793152, 'test_accuracy': 0.6666666666666666, 'test_precision': 0.6666666666666666, 'test_recall': 1.0, 'test_f1_score': 0.8, 'test_roc_auc': 0.58, 'test_false_positives_rate': 1.0, 'test_false_negatives_rate': 0.0, 'test_runtime': 0.3025, 'test_samples_per_second': 49.591, 'test_steps_per_second': 3.306, 'test_epoch': 1.0}
2025-05-16 02:30:22,066 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 02:30:22,071 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 02:30:22,138 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 02:30:22,140 - Experiment - INFO - Finished model evaluations stage.


### EUvsDisinfo Dataset

In [ ]:
dataset_name = 'EUvsDisinfo'
dataset_path = '../../sources/local_datasets/data/EUvsDisinfo/euvsdisinfo.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          dataset_hparams_grid,
                          model_hparams_grid)

### ISOT Dataset

In [ ]:
dataset_name = 'ISOT'
dataset_path = '../../sources/local_datasets/data/ISOT/isot.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          dataset_hparams_grid,
                          model_hparams_grid)

### CZ Dataset

In [5]:
cz_pipelines = []

for max_tokens in max_tokens_params:
    cz_pipelines.extend([PreprocessingPipeline(name='minimal',
                                               iterable=[Encoder(truncation_max_length=max_tokens)]),
                         PreprocessingPipeline(name='noise_reduction',
                                               iterable=[NoiseReduction(),
                                                         Encoder(truncation_max_length=max_tokens)]),
                         PreprocessingPipeline(name='noise_reduction_with_lemmatization',
                                               iterable=[NoiseReduction(), NLTKTokenizer(language='czech'),
                                                         Lemmatization(language='czech'),
                                                         Encoder(is_split_into_words=True,
                                                                 truncation_max_length=max_tokens)]),
                         PreprocessingPipeline(name='noise_reduction_with_stemming',
                                               iterable=[NoiseReduction(), NLTKTokenizer(language='czech'),
                                                         Stemming(language='czech'),
                                                         Encoder(is_split_into_words=True,
                                                                 truncation_max_length=max_tokens)])
                         ])
# optional - for testing purposes, if you want to run fast test on very small datasets
truncate(cz_pipelines, fraction=0.1)

cz_dataset_hparams_grid = ParameterGrid({'preprocessings_pipeline': cz_pipelines})
print('Dataset Hyperparameters Grid Size: ', len(cz_dataset_hparams_grid))
print('Model Hyperparameters Grid Size: ', len(model_hparams_grid))
print('Total Hyperparameter Combinations: ', len(cz_dataset_hparams_grid) * len(model_hparams_grid))

Dataset Hyperparameters Grid Size:  4
Model Hyperparameters Grid Size:  1
Total Hyperparameter Combinations:  4


In [6]:
dataset_name = 'cz_dataset'
dataset_path = '../../sources/local_datasets/data/cz_dataset/cz_dataset.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          cz_dataset_hparams_grid,
                          model_hparams_grid)

### Run 1 of 4

2025-05-16 01:23:04,217 - Experiment - INFO - MLflow experiment initialized with ID: 102456789287063802
2025-05-16 01:23:04,218 - Experiment - INFO - Model signature: model(mn=bert,ti=bert-base-uncased,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01)
2025-05-16 01:23:04,219 - Experiment - INFO - Dataset signature: dataset(dn=cz_dataset,tp=0.7,vp=0.15);pipeline([truncation(f=0.1,tt=top),bert_base_uncased_encoder(t=bert-base-uncased,t=true,p=max_length,tml=128,isiw=false)])
2025-05-16 01:23:04,220 - Experiment - INFO - Run signature: model(mn=bert,ti=bert-base-uncased,mmn=loss,e=10,bs=16,ebs=16,lr=5e-05,ls=linear,wd=0.001,o=adamw_torch,esp=3,est=0.01);dataset(dn=cz_dataset,tp=0.7,vp=0.15);pipeline([truncation(f=0.1,tt=top),bert_base_uncased_encoder(t=bert-base-uncased,t=true,p=max_length,tml=128,isiw=false)])
2025-05-16 01:23:05,092 - Experiment - INFO - Run ID: 1fe982d9e2884fb5b8a1f68edfc38268
2025-05-16 01:23:05,092 - Experiment - INFO - Run name: mo

Map:   0%|          | 0/69 [00:00<?, ? examples/s]

Map:   0%|          | 0/69 [00:00<?, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

...

2025-05-16 01:25:20,364 - Experiment - INFO - Test metrics: {'test_loss': 0.6642666459083557, 'test_accuracy': 0.6666666666666666, 'test_precision': 1.0, 'test_recall': 0.5454545454545454, 'test_f1_score': 0.7058823529411765, 'test_roc_auc': 0.9772727272727273, 'test_false_positives_rate': 0.0, 'test_false_negatives_rate': 0.45454545454545453, 'test_runtime': 0.1551, 'test_samples_per_second': 96.693, 'test_steps_per_second': 6.446, 'test_epoch': 3.0}
2025-05-16 01:25:20,378 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:25:20,381 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:25:20,419 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:25:20,420 - Experiment - INFO - Evaluating model using f1_score metric...
2025-05-16 01:25:20,421 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-15
2025-05-16 01:25:20,857 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.6568316

2025-05-16 01:25:21,013 - Experiment - INFO - Test metrics: {'test_loss': 0.6642666459083557, 'test_accuracy': 0.6666666666666666, 'test_precision': 1.0, 'test_recall': 0.5454545454545454, 'test_f1_score': 0.7058823529411765, 'test_roc_auc': 0.9772727272727273, 'test_false_positives_rate': 0.0, 'test_false_negatives_rate': 0.45454545454545453, 'test_runtime': 0.1485, 'test_samples_per_second': 101.02, 'test_steps_per_second': 6.735, 'test_epoch': 3.0}
2025-05-16 01:25:21,031 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:25:21,033 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:25:21,086 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:25:21,087 - Experiment - INFO - Evaluating model using false_positives_rate metric...
2025-05-16 01:25:21,088 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-15
2025-05-16 01:25:21,594 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss

2025-05-16 01:25:21,733 - Experiment - INFO - Test metrics: {'test_loss': 0.6642666459083557, 'test_accuracy': 0.6666666666666666, 'test_precision': 1.0, 'test_recall': 0.5454545454545454, 'test_f1_score': 0.7058823529411765, 'test_roc_auc': 0.9772727272727273, 'test_false_positives_rate': 0.0, 'test_false_negatives_rate': 0.45454545454545453, 'test_runtime': 0.1293, 'test_samples_per_second': 115.982, 'test_steps_per_second': 7.732, 'test_epoch': 3.0}
2025-05-16 01:25:21,759 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:25:21,763 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:25:21,817 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:25:21,818 - Experiment - INFO - Evaluating model using roc_auc metric...
2025-05-16 01:25:21,819 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-15
2025-05-16 01:25:22,229 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.6568316

2025-05-16 01:25:22,355 - Experiment - INFO - Test metrics: {'test_loss': 0.6642666459083557, 'test_accuracy': 0.6666666666666666, 'test_precision': 1.0, 'test_recall': 0.5454545454545454, 'test_f1_score': 0.7058823529411765, 'test_roc_auc': 0.9772727272727273, 'test_false_positives_rate': 0.0, 'test_false_negatives_rate': 0.45454545454545453, 'test_runtime': 0.1209, 'test_samples_per_second': 124.08, 'test_steps_per_second': 8.272, 'test_epoch': 3.0}
2025-05-16 01:25:22,369 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:25:22,371 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:25:22,409 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:25:22,410 - Experiment - INFO - Evaluating model using loss metric...
2025-05-16 01:25:22,410 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-15
2025-05-16 01:25:22,812 - Experiment - INFO - Best entry according to validation metrics: {'eval_loss': 0.65683168172

2025-05-16 01:25:22,950 - Experiment - INFO - Test metrics: {'test_loss': 0.6642666459083557, 'test_accuracy': 0.6666666666666666, 'test_precision': 1.0, 'test_recall': 0.5454545454545454, 'test_f1_score': 0.7058823529411765, 'test_roc_auc': 0.9772727272727273, 'test_false_positives_rate': 0.0, 'test_false_negatives_rate': 0.45454545454545453, 'test_runtime': 0.1291, 'test_samples_per_second': 116.209, 'test_steps_per_second': 7.747, 'test_epoch': 3.0}
2025-05-16 01:25:22,974 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-16 01:25:22,977 - Experiment - INFO - Saving evaluation data in json...
2025-05-16 01:25:23,035 - Experiment - INFO - Successfully saved evaluation data.
2025-05-16 01:25:23,036 - Experiment - INFO - Finished model evaluations stage.
